In [2]:
import polars as pl
from pathlib import Path
from datetime import date, timedelta
import time

In [3]:
routes = ["M1", "M2", "M4", "M15", "M101"]
 
DATE_START = date(2026, 3, 3)
DATE_END = date(2026, 7, 3)
 
TRIP_UPDATES_B64 = "aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL3RyaXBVcGRhdGVz"
VEHICLE_POS_B64 = "aHR0cHM6Ly9ndGZzcnQucHJvZC5vYmFueWMuY29tL3ZlaGljbGVQb3NpdGlvbnM"
 
TRIP_UPDATES_DIR = Path("raw/trip_updates")
VEHICLE_POS_DIR = Path("raw/vehicle_positions")
TRIP_UPDATES_DIR.mkdir(parents=True, exist_ok=True)
VEHICLE_POS_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
def rt_url(kind: str, d: date, b64: str) -> str:
    return f"http://parquet.gtfsrt.io/{kind}/date={d.isoformat()}/base64url={b64}/data.parquet"
 
 
def fetch_trip_updates(d: date) -> bool:
    out_path = TRIP_UPDATES_DIR / f"{d.isoformat()}.parquet"
    if out_path.exists():
        return True  # already have it, skip (resumable)
    try:
        df = (
            pl.scan_parquet(rt_url("trip_updates", d, TRIP_UPDATES_B64))
            .filter(pl.col("route_id").is_in(routes))
            .select([
                "feed_timestamp", "fetch_timestamp", "trip_id", "route_id",
                "direction_id", "start_time", "start_date", "schedule_relationship",
                "vehicle_id", "trip_timestamp", "stop_sequence", "stop_id",
                "arrival_time", "departure_time",
            ])
            .collect(engine="streaming")
        )
        df.write_parquet(out_path)
        print(f"  trip_updates {d}: {df.height:,} rows")
        return True
    except Exception as e:
        print(f"  trip_updates {d}: FAILED - {e}")
        return False
 
 
def fetch_vehicle_positions(d: date) -> bool:
    out_path = VEHICLE_POS_DIR / f"{d.isoformat()}.parquet"
    if out_path.exists():
        return True
    try:
        df = (
            pl.scan_parquet(rt_url("vehicle_positions", d, VEHICLE_POS_B64))
            .filter(pl.col("route_id").is_in(routes))
            .select([
                "trip_id", "route_id", "vehicle_id", "direction_id",
                "start_date", "latitude", "longitude", "bearing",
                "stop_id", "timestamp",
            ])
            .collect(engine="streaming")
        )
        df.write_parquet(out_path)
        print(f"  vehicle_positions {d}: {df.height:,} rows")
        return True
    except Exception as e:
        print(f"  vehicle_positions {d}: FAILED - {e}")
        return False

In [5]:
dates = [DATE_START + timedelta(days=i) for i in range((DATE_END - DATE_START).days + 1)]
 
failed_dates = []
for d in dates:
    print(f"=== {d} ===")
    ok_tu = fetch_trip_updates(d)
    ok_vp = fetch_vehicle_positions(d)
    if not (ok_tu and ok_vp):
        failed_dates.append(d)
    time.sleep(0.2)  # light throttle, avoid hammering the endpoint
 
print(f"\nDone. {len(dates) - len(failed_dates)}/{len(dates)} days fully fetched.")
if failed_dates:
    print("Failed/incomplete dates (re-run this script to retry just these - already-saved days are skipped):")
    for d in failed_dates:
        print(" -", d)

=== 2026-03-03 ===
=== 2026-03-04 ===
=== 2026-03-05 ===
  trip_updates 2026-03-05: 8,255,770 rows
  vehicle_positions 2026-03-05: 214,747 rows
=== 2026-03-06 ===
  trip_updates 2026-03-06: 8,218,991 rows
  vehicle_positions 2026-03-06: 214,919 rows
=== 2026-03-07 ===
  trip_updates 2026-03-07: 6,859,973 rows
  vehicle_positions 2026-03-07: 155,788 rows
=== 2026-03-08 ===
  trip_updates 2026-03-08: 6,490,689 rows
  vehicle_positions 2026-03-08: 140,177 rows
=== 2026-03-09 ===
  trip_updates 2026-03-09: 7,847,005 rows
  vehicle_positions 2026-03-09: 201,771 rows
=== 2026-03-10 ===
  trip_updates 2026-03-10: 8,169,839 rows
  vehicle_positions 2026-03-10: 211,247 rows
=== 2026-03-11 ===
  trip_updates 2026-03-11: 8,286,493 rows
  vehicle_positions 2026-03-11: 215,361 rows
=== 2026-03-12 ===
=== 2026-03-13 ===
  trip_updates 2026-03-13: 8,027,684 rows
  vehicle_positions 2026-03-13: 208,540 rows
=== 2026-03-14 ===
  trip_updates 2026-03-14: 7,198,999 rows
  vehicle_positions 2026-03-14: 16